# Part 1. Prompt Engineering

- A **prompt** is a **set of instructions** (including content   like text, images, etc) provided to an LLM to perform a task.

- **Prompt Engineering** is the process of crafting such the set of instructions that gets an LLM model to generate the desired outcome (i.e., solving the task). 

#### Components of Well-Structured Prompts
- **Role**: The role the LLM should adopt.
- **Task Description**: The specific instruction or question.
- **Context**: Additional information needed for the task.
- **Output Format**: How the response should be structured.
- **Examples (a.k.a. Few-Shot "Learning")** (optional): Sample input/output pairs.

For the "Prompt Engineering and Structured Outputs" section of our tutorial, we will rely on OpenAI Software Development Kit (OpenAI SDK), which can be installed by running `pip install openai` (or similar).
We will use models running within Ollama and OpenAI SDK to connect to Ollama.

In [1]:
#### UNCOMMENT ONE OF THE FOLLOWING:

### OPTION 1 - Running ollama locally

# When running ollama locally on your computer use the following command 
# to check the exact model names: ! ollama list.

llama8b = "llama3.1:latest"
llama3b = "llama3.2:3b"

### The end of OPTION 1.

############################################################################

### OPTION 2 - TACC Analysis Portal (https://tap.tacc.utexas.edu):

# # Start our ollama server in the background to host our LLMs
# from ollama_utils import start_ollama_server, stop_ollama_server

# # start ollama server
# start_ollama_server()

# # model names
# llama8b = "llama3.1:8b"
# llama3b = "llama3.2:latest"

### The end of OPTION 2.

In [2]:
# Import OpenAI client class
from openai import OpenAI

# Import other modules
import textwrap

In [5]:
# Connect to Ollama running on the backend

openai_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

In [3]:
# Define a wrapper function that will send requests to an LLM and receive responses
def generate_response(
        client: OpenAI,
        model: str,
        user_prompt: str,
        system_prompt: str = "You are a helpful assistant.",
        temperature: float = 0.5       
) -> str:
    """Sends a request to an LLM and and returns a response."""
    
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.choices[0].message.content

#### 1. Vague vs. Precise Prompt
Let's see how an LLM's output changes when we start from a vague prompt, then adding a persona/role, and finally specifying the task.

In [6]:
# Vague prompt = vague result.
# We should expect a generic output from the LLM.
vague_prompt = "Tell me about Albert Einstein."

vague_response = generate_response(
    client=openai_client,
    model=llama8b,
    user_prompt=vague_prompt, 
)

print(vague_response)
# textwrap.fill(vague_response, width=80)

Albert Einstein (1879-1955) was a renowned German-born physicist who revolutionized our understanding of space, time, and gravity. He is widely regarded as one of the most influential scientists of the 20th century.

**Early Life and Education**

Einstein was born in Munich, Germany, to a Jewish family. He grew up in a middle-class household and was an average student in school. However, he was fascinated by science and mathematics from an early age. He spent much of his childhood playing the violin and reading extensively on various subjects, including physics and mathematics.

Einstein studied physics at the Swiss Federal Polytechnic University, where he graduated in 1900. He then worked as a patent clerk in Bern, Switzerland, where he developed his theory of special relativity.

**Theories and Contributions**

Einstein's work had a profound impact on our understanding of the universe. Some of his most significant contributions include:

1. **Special Relativity (1905)**: Einstein's t

In [7]:
# Add role (persona) to the vague prompt.
# The tone of the output should match the role provided in the prompt.
vague_system_prompt_with_persona = "You are a high-school physics teacher."
vague_user_prompt_with_persona = "Tell me about Albert Einstein."


response_with_persona = generate_response(
    client=openai_client,
    model=llama8b,
    system_prompt=vague_system_prompt_with_persona,
    user_prompt=vague_user_prompt_with_persona,
)

print(response_with_persona)
# textwrap.fill(response_with_persona, width=80)

Albert Einstein! One of the greatest minds in the history of physics. My students love learning about him, and I'm happy to share some of his fascinating story with you.

Born in 1879 in Germany, Einstein grew up with a curious mind and a passion for learning. His early life was marked by a love for music, reading, and exploring the natural world. He was a bit of a rebellious thinker, often challenging traditional ideas and questioning authority.

Einstein's major contribution to physics came in the form of his theory of relativity. He proposed that time and space are not absolute, but are relative to the observer. His famous equation, E=mc², shows that energy and mass are interchangeable. This idea revolutionized our understanding of space, time, and gravity.

But Einstein's work didn't stop there. He also made significant contributions to the development of quantum mechanics, the photoelectric effect, and the behavior of light. His theories challenged the long-held Newtonian views of

In [8]:
# Finally let's narrow down the scope of the output by providing 
# specific task and the output format.
precise_system_prompt = "You are a high-school physics teacher."

precise_user_prompt = """
Explain the role Albert Einstein played in the development of physics 
in 20th century.
Write two paragraphs.
"""

precise_response = generate_response(
    client=openai_client,
    model=llama8b,
    system_prompt=precise_system_prompt,
    user_prompt=precise_user_prompt
)

print(precise_response)
# textwrap.fill(precise_response, width=80)

Albert Einstein played a pivotal role in the development of physics in the 20th century. His groundbreaking work in the early 20th century revolutionized our understanding of space, time, and gravity. In 1905, Einstein's theory of special relativity posited that time and space are relative, and that the laws of physics are the same for all observers in uniform motion. This theory challenged the long-held notion of absolute time and space, and it had far-reaching implications for our understanding of the universe. In 1915, Einstein expanded his theory to include gravity, introducing the concept of general relativity, which describes gravity as the curvature of spacetime caused by massive objects.

Einstein's work had a profound impact on the development of modern physics. His theories predicted phenomena such as gravitational waves and black holes, which were later confirmed by observations and experiments. His work also laid the foundation for the development of quantum mechanics and t

In the case of the vague prompt, we can see that the output text covers various aspects of Albert Einstein's live without providing depth into any of those. Advancing forward, when we provide a role ("a high-school physics teacher"), the output changes its format and is "tailored" to the role/persona specified. 
Finally, with the precise prompt, we **guide** an LLM to produce a specific but more detailed response. While for some cases (e.g., famous personas) we might want to learn both perspectives (in breadth and depth), for AI applications we typically want to narrow down the scope of the task to get more specific outputs.

So, **Best Practice #1: Be specific and direct.**

#### 2. Adding Context
Any LLM model "knows" only what it was trained on.
However, you can incorporate new information by providing 
it as a context.

In [9]:
# Source: https://tacc.utexas.edu/about/staff-directory/niall-gaffney/
context = """
NIALL GAFFNEY

Data and AI Directorate

Phone: 512-475-9504 | Email: ngaffney@tacc.utexas.edu

Education: B.A., M.A., Ph.D., Astronomy University of Texas at Austin

Niall Gaffney's background primarily revolves around the management and utilization of 
large inhomogeneous scientific datasets. Niall, who earned his B.A., M.A., and Ph.D. 
degrees in astronomy from The University of Texas at Austin, joined TACC in May 2013. 
Most of his focus has been on creating environments to foster better data practices 
from improving metadata, data processing, analysis, and reuse. He focuses on improving 
researchers' data practices to accelerate outcomes and better feed the Machine Learning 
and Artificial Intelligence applications which are becoming more broadly adopted in 
science and engineering research fields. Much of this stems from his 13 years as designer 
and developer for the archives at the Space Telescope Science Institute (STScI), which 
holds the data from the Hubble Space Telescope, Kepler, and James Webb Space Telescope 
missions.

He was also a leader in developing the Hubble Legacy Archive. This project harvested 
the 20+ years of Hubble Space Telescope data to create some of the most sensitive 
astronomical data products available for open research. Before his work at STScI, 
Niall was worked as "the friend of the telescope" for the Hobby Eberly Telescope (HET) 
project at the McDonald Observatory in west Texas. This was the start of his work in 
planning experiments and then cataloging the data the HET produced.
"""

**Temperature** is the parameter for controlling the randomness (or creativity) of a model. By increasing the temperature from 0 to 1, we're increasing the randomness (hence more creativity) of the response. 

In [18]:
# First let's see what a model "knows" about Niall Gaffney.
# In other words, do NOT provide any context first. 

# Tip: Modify temperature and see how this parameter influences
# the output of the model by sending requests multiple times.
no_context_prompt = "Who is Niall Gaffney?"

response_wo_context = generate_response(
    client=openai_client,
    model=llama8b,
    user_prompt=no_context_prompt,
    temperature=0.0
)

print(response_wo_context)

Niall Gaffney is a British computer scientist and data scientist. He is the Director of the UK's Office for Artificial Intelligence (OAI) and has been instrumental in shaping the UK's AI strategy.


In [19]:
# Now, let's add the context.
# Tip: run this part multiple times for temperature=0.0 and temperature=1.0
prompt_with_context = no_context_prompt + f"\nContext: {context}"

response_with_context = generate_response(
    client=openai_client,
    model=llama8b,
    user_prompt=prompt_with_context,
    temperature=0.0
)

print(response_with_context)

Niall Gaffney is a data and AI expert with a background in astronomy. He has a Ph.D. in astronomy from the University of Texas at Austin and has worked in various roles related to managing and utilizing large scientific datasets. Specifically, he has:

* Worked as a designer and developer for the archives at the Space Telescope Science Institute (STScI), where he was involved in managing data from several major space missions, including the Hubble Space Telescope, Kepler, and James Webb Space Telescope.
* Led the development of the Hubble Legacy Archive, which harvested 20+ years of Hubble Space Telescope data to create sensitive astronomical data products for open research.
* Worked as "the friend of the telescope" for the Hobby Eberly Telescope (HET) project at the McDonald Observatory in west Texas, where he planned experiments and cataloged data.

Currently, Niall Gaffney is the Director of the Data and AI Directorate at the Texas Advanced Computing Center (TACC), where he focuses 

We've seen that by providing context, we can "add" new information/knowledge to a model without retraining it. What's more, by providing context we can also reduce the occurrence of hallucinations [1]. However, note that adding the context **does NOT guarantee** that a model will strictly follow it [1].

Also, adding additional instructions like "generate response based on the context provided" to the prompt can also be helpful.

**Best Practice #2: Add specific context to your prompts when applicable.**

[1] Huyen, C. (2024). Prompt Engineering. In AI Engineering: Building Applications with Foundation Models. (pp. 211-252) O'Reilly Media, Inc.

#### 3. Prompt Chaining
Break complex tasks into simpler subtasks.
Use each LLM's output as a **context** for the next prompt/step.

In [20]:

topic = "the impact of Albert Einstein on philosophy of 20th century"

# Step 1: Create an outline for the blog post
prompt_step_1 = f"Come up with a 3 point outline for a post about: {topic}"

response_step_1 = generate_response(
    client=openai_client,
    model=llama8b,
    user_prompt=prompt_step_1,
    temperature=0.7
)

print(response_step_1)

Here's a 3-point outline for a post on the impact of Albert Einstein on the philosophy of the 20th century:

**I. Challenging Traditional Notions of Space and Time**

* Einstein's theory of relativity (1905, 1915) revolutionized our understanding of space and time, challenging the long-held notion of an absolute, fixed space and time.
* This challenge had far-reaching implications for philosophy, particularly in the areas of metaphysics and epistemology, where the nature of reality and knowledge was re-examined.

**II. Influencing the Development of Post-Modern and Contemporary Philosophy**

* Einstein's theory of relativity and his thoughts on the nature of space and time influenced the development of postmodern and contemporary philosophical movements, such as:
	+ Phenomenology (e.g., Maurice Merleau-Ponty, 1908-1961)
	+ Existentialism (e.g., Martin Heidegger, 1889-1976)
	+ Poststructuralism (e.g., Jacques Derrida, 1930-2004)

**III. Shaping the Philosophy of Science and the Nature o

In [21]:
# Step 2: Write introduction using the outline
prompt_step_2=f"""
Using the following OUTLINE, write an introduction paragraph with 80-100 words.
OUTLINE: {response_step_1}
Hook the reader with a surprising fact in the first sentence.
"""

response_step_2 = generate_response(
    client=openai_client,
    model=llama8b,
    user_prompt=prompt_step_2,
    temperature=0.7
)

print(response_step_2)

Before Albert Einstein's groundbreaking theory of relativity was even published, the famous physicist had already been warned by a colleague that his ideas would "upset the whole of physics." But what Einstein's colleague didn't anticipate was the profound impact his theory would have on the philosophy of the 20th century. Einstein's challenge to traditional notions of space and time had far-reaching implications for philosophy, particularly in the areas of metaphysics and epistemology. His revolutionary ideas would go on to influence some of the most significant philosophical movements of the 20th century.


In [22]:
# Step 3: Come up with a few options for the title
prompt_step_3 = f"""
Based on the INTRODUCTION below, come up with 3 catchy blog post titles.
INTRODUCTION: {response_step_2}
Format your output as 3 bullet points.
""" 
response_step_3 = generate_response(
    client=openai_client,
    model=llama8b,
    user_prompt=prompt_step_3,
    temperature=1
)

print(response_step_3)

Here are 3 catchy blog post title options:

• "The Ripple Effect: How Einstein's Theory Shook the Foundations of Philosophy"
• "Relativity Beyond Physics: Unpacking the Unforeseen Implications for Philosophy"
• "Time to Rethink: How Einstein's Groundbreaking Theory Redefined the Boundaries of Science and Philosophy"


Main motivation behind the prompt chaining technique: a few smaller prompts are better than one that is a giant one. If your application require solving complex tasks with multiple steps, divide them into smaller subtasks by introducing smaller prompts and chain LLM's outputs together with the following prompts.

Remember: When working with LLMs, simpler instructions are better than complex ones.

**Best Practice #3: Break complex prompts into smaller ones.**

#### 4. More Practice - Code Generation

In [23]:
role = "a Python Developer"
task_description = "to develop Python code based on the specification provided"
context = "write a Python function that generates Fibonacci sequence; use a `while` loop"
output_format = """\n
    **EXPLANATION**: [provide a brief explanation of your solution]
    **PYTHON CODE**: [a block of Python code]
"""

# Do not provide additional text after the **PYTHON CODE** section

In [27]:
system_prompt = f"You are {role}. Your task is {task_description}."
user_prompt = f"""
Here are additional instructions: {context}.
OUTPUT FORMAT: {output_format}
"""

print("SYSTEM PROMPT:", system_prompt, sep="\n")
print("-" * 100)
print("USER PROMPT:", user_prompt, sep="\n")

SYSTEM PROMPT:
You are a Python Developer. Your task is to develop Python code based on the specification provided.
----------------------------------------------------------------------------------------------------
USER PROMPT:

Here are additional instructions: write a Python function that generates Fibonacci sequence; use a `while` loop.
OUTPUT FORMAT: 

    **EXPLANATION**: [provide a brief explanation of your solution]
    **PYTHON CODE**: [a block of Python code]




In [28]:
response = generate_response(
    client=openai_client,
    model=llama8b,
    user_prompt=user_prompt,
    system_prompt=system_prompt,
)

print(response)

**EXPLANATION**: 
The Fibonacci sequence is a series of numbers where a number is the addition of the last two numbers, starting with 0 and 1. This function will generate the Fibonacci sequence up to a given number of terms.

**PYTHON CODE**:

```python
def generate_fibonacci(n):
    """
    Generate the Fibonacci sequence up to n terms.

    Args:
        n (int): The number of terms to generate.

    Returns:
        list: A list of the first n Fibonacci numbers.
    """
    fib_sequence = [0, 1]
    while len(fib_sequence) < n:
        fib_sequence.append(fib_sequence[-1] + fib_sequence[-2])
    return fib_sequence

# Example usage:
n = 10  # Generate the first 10 Fibonacci numbers
print(generate_fibonacci(n))
```

This function starts with a list containing the first two Fibonacci numbers, [0, 1]. Then, it enters a `while` loop that continues until the list has reached the desired number of terms. Inside the loop, it appends the sum of the last two numbers in the list to the end of